# Stable Diffusion 1.5 FastAPI Backend on Google Colab

This notebook runs a FastAPI server hosting Stable Diffusion 1.5 on a free Colab GPU and exposes it to the internet via ngrok.

---

## 📋 Quickstart Instructions

1. **Enable GPU Acceleration**:
   - In the top menu, select **Runtime** > **Change runtime type**.
   - Set **Hardware accelerator** to **T4 GPU** (or any available GPU) and click **Save**.

2. **Install Dependencies**:
   - Run **Cell 1** below to install all required packages.

3. **Configure Your ngrok Auth Token**:
   - Sign up / log in to [ngrok](https://dashboard.ngrok.com/).
   - Copy your authtoken from [ngrok authtoken dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
   - In **Cell 2**, paste your token into: `NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"`.

4. **Start the Server**:
   - Run **Cell 2**. This will download Stable Diffusion 1.5, start the FastAPI server, and open an ngrok tunnel.

5. **Connect from Your Local Machine**:
   - Look at the output of Cell 2 for your public URL:
     ```text
     Public ngrok URL: https://xxxx-xx-xx-xx-xx.ngrok-free.app
     Target endpoint:  https://xxxx-xx-xx-xx-xx.ngrok-free.app/generate
     ```
   - Copy this URL and paste it into `NGROK_URL` inside your local `client.py`.
   - Keep this Colab tab open and running while making requests from your local machine.

In [ ]:
# Step 1: Install required packages
!pip install diffusers transformers accelerate fastapi uvicorn pyngrok nest-asyncio

### Step 2: Load Model, Start FastAPI & Launch ngrok Tunnel
Remember to update `NGROK_AUTH_TOKEN` below before running this cell.

In [ ]:
import io
import torch
from diffusers import StableDiffusionPipeline
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
import nest_asyncio
from google.colab import userdata

# 1. Load Stable Diffusion 1.5 model onto GPU
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model '{model_id}' onto {device}...")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
pipe = pipe.to(device)
print("Model loaded successfully!")

# 2. Initialize FastAPI app and define request schema
app = FastAPI(title="Stable Diffusion Backend")

class PromptRequest(BaseModel):
    prompt: str
    num_inference_steps: int = 30
    guidance_scale: float = 7.5

@app.post("/generate")
def generate_image(request: PromptRequest):
    try:
        image = pipe(
            request.prompt,
            num_inference_steps=request.num_inference_steps,
            guidance_scale=request.guidance_scale
        ).images[0]

        # Save image to in-memory bytes buffer
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
        buffer.seek(0)

        return Response(content=buffer.getvalue(), media_type="image/png")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 3. Configure ngrok authentication and start tunnel
# Replace 'YOUR_NGROK_AUTH_TOKEN_HERE' with your actual token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url
print("=" * 60)
print(f"Public ngrok URL: {public_url}")
print(f"Target endpoint:  {public_url}/generate")
print("=" * 60)

# 4. Apply nest_asyncio to allow running uvicorn inside Jupyter/Colab event loop
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)
